# 🇹🇷 Sıfırdan LLM Eğitimi — 6B Parametre

**Hedef:** 2 saat H100 ile Türkçe + İngilizce LLM eğitimi

**Mimari:** Llama (Dense Transformer) — 350M parametre

**Adımlar:**
1. Sistem kontrolü & kütüphane kurulumu
2. Google Drive bağlantısı
3. Veri indirme (Türkçe Wikipedia + İngilizce FineWeb)
4. Tokenizasyon
5. Model oluşturma (sıfırdan)
6. Eğitim (~2 saat)
7. Kaydetme & değerlendirme
8. HuggingFace Hub + GitHub'a yükleme

> ⚠️ **Runtime > Change runtime type > H100 GPU** seçin!

In [ ]:
#@title 🔍 BÖLÜM 1: Sistem Kontrolü
print("=" * 60)
print("🔍 BÖLÜM 1: Sistem Kontrolü")
print("=" * 60)

import subprocess
import sys
import os

# GPU kontrolü
gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu_info.stdout)

import torch
assert torch.cuda.is_available(), "❌ CUDA bulunamadı!"

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"✅ GPU: {gpu_name}")
print(f"✅ VRAM: {gpu_mem_gb:.0f} GB")
print(f"✅ CUDA: {torch.version.cuda}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ BF16: {torch.cuda.is_bf16_supported()}")

if "H100" not in gpu_name and "A100" not in gpu_name:
    print("⚠️  H100/A100 değil! Eğitim yavaş olabilir.")
    print("   Runtime > Change runtime type > GPU: H100 seçin")

In [ ]:
#@title 📦 BÖLÜM 2: Kütüphane Kurulumu
print("\n" + "=" * 60)
print("📦 BÖLÜM 2: Kütüphane Kurulumu")
print("=" * 60)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.46.0",
    "accelerate>=0.34.0",
    "datasets>=3.0.0",
    "tokenizers>=0.19.0",
    "trl>=0.11.0",
    "wandb",
    "huggingface_hub>=0.24.0",
    "kagglehub",
    "datasketch",
    "langdetect",
    "sentencepiece",
    "bitsandbytes>=0.43.0",
], check=True)

# Flash Attention (ayrı kur, build gerekebilir)
try:
    import flash_attn
    print(f"✅ Flash Attention {flash_attn.__version__} zaten kurulu")
except ImportError:
    print("📦 Flash Attention kuruluyor (bu birkaç dakika sürebilir)...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "flash-attn>=2.6.3", "--no-build-isolation",
    ])

print("✅ Tüm kütüphaneler kuruldu!")

# API Keyler
# ⚠️ Secret/token değerlerini notebook içine KESİNLİKLE gömmeyin.
# Colab: soldaki "Secrets" panelinden ekleyin veya ortam değişkeni olarak set edin.
# Yerel: export KAGGLE_USERNAME=... ; export KAGGLE_KEY=...
os.environ["KAGGLE_USERNAME"] = os.getenv("KAGGLE_USERNAME", "")
os.environ["KAGGLE_KEY"] = os.getenv("KAGGLE_KEY", "")
if os.environ["KAGGLE_USERNAME"] and os.environ["KAGGLE_KEY"]:
    print("✅ Kaggle API env üzerinden ayarlandı")
else:
    print("⚠️  KAGGLE_USERNAME/KAGGLE_KEY bulunamadı (env/Secrets). Gerekirse ekleyin.")

In [ ]:
#@title 💾 BÖLÜM 3: Google Drive Bağlantısı
print("\n" + "=" * 60)
print("💾 BÖLÜM 3: Google Drive Bağlantısı")
print("=" * 60)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/LLM_Training"
    IN_COLAB = True
except ImportError:
    SAVE_DIR = "./output"
    IN_COLAB = False
    print("⚠️  Colab ortamında değilsiniz, yerel dizin kullanılıyor")

CHECKPOINT_DIR = os.path.join(SAVE_DIR, "checkpoints")
MODEL_DIR = os.path.join(SAVE_DIR, "final_model")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"✅ Checkpoint dizini: {CHECKPOINT_DIR}")

In [ ]:
#@title 📥 BÖLÜM 4: Veri İndirme
print("\n" + "=" * 60)
print("📥 BÖLÜM 4: Veri İndirme")
print("=" * 60)

from datasets import load_dataset, Dataset, concatenate_datasets
import random

random.seed(42)

NUM_SAMPLES = 200_000  # 2 saat için yeterli

all_texts = []

# --- Türkçe Wikipedia ---
print("📥 Türkçe Wikipedia indiriliyor...")
try:
    wiki_tr = load_dataset(
        "wikimedia/wikipedia", "20231101.tr",
        split=f"train[:{NUM_SAMPLES}]",
        trust_remote_code=True,
    )
    for item in wiki_tr:
        if len(item["text"]) > 200:
            all_texts.append({"text": item["text"][:5000], "lang": "tr"})
    print(f"  ✅ {sum(1 for t in all_texts if t['lang']=='tr'):,} Türkçe belge")
except Exception as e:
    print(f"  ⚠️ Wikipedia-TR: {e}")

# --- İngilizce FineWeb-Edu ---
print("📥 İngilizce veri indiriliyor...")
try:
    fineweb = load_dataset(
        "HuggingFaceFW/fineweb-edu", "sample-10BT",
        split=f"train[:{int(NUM_SAMPLES * 1.5)}]",
        trust_remote_code=True,
    )
    for item in fineweb:
        if len(item["text"]) > 200:
            all_texts.append({"text": item["text"][:5000], "lang": "en"})
    print(f"  ✅ {sum(1 for t in all_texts if t['lang']=='en'):,} İngilizce belge")
except Exception as e:
    print(f"  ⚠️ FineWeb: {e}")
    # Alternatif
    try:
        wikitext = load_dataset("wikitext", "wikitext-103-raw-v1", split=f"train[:{NUM_SAMPLES}]")
        for item in wikitext:
            if len(item["text"]) > 200:
                all_texts.append({"text": item["text"][:5000], "lang": "en"})
        print(f"  ✅ WikiText alternatif kullanıldı")
    except:
        pass

print(f"\n📊 Toplam: {len(all_texts):,} belge")
tr_count = sum(1 for t in all_texts if t["lang"] == "tr")
en_count = sum(1 for t in all_texts if t["lang"] == "en")
print(f"  TR: {tr_count:,} | EN: {en_count:,}")

# Karıştır
random.shuffle(all_texts)
raw_dataset = Dataset.from_list(all_texts)

In [ ]:
#@title 🔤 BÖLÜM 5: Tokenizasyon
print("\n" + "=" * 60)
print("🔤 BÖLÜM 5: Tokenizasyon")
print("=" * 60)

from transformers import AutoTokenizer
from itertools import chain

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

SEQ_LENGTH = 2048
VOCAB_SIZE = tokenizer.vocab_size

print(f"  Tokenizer: GPT-2 ({VOCAB_SIZE} vocab)")
print(f"  Sequence uzunluğu: {SEQ_LENGTH}")

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=False, padding=False,
                     return_attention_mask=False)

tokenized = raw_dataset.map(tokenize_fn, batched=True, batch_size=1000,
                             num_proc=2, remove_columns=raw_dataset.column_names)

def group_texts(examples):
    concatenated = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // SEQ_LENGTH) * SEQ_LENGTH
    result = {
        k: [concatenated[k][i:i + SEQ_LENGTH] for i in range(0, total_length, SEQ_LENGTH)]
        for k in concatenated.keys()
    }
    result["labels"] = result["input_ids"].copy()
    return result

grouped = tokenized.map(group_texts, batched=True, batch_size=1000, num_proc=2)

split = grouped.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

total_tokens = len(train_dataset) * SEQ_LENGTH
print(f"\n✅ Train: {len(train_dataset):,} sequence ({total_tokens:,} token = {total_tokens/1e6:.1f}M)")
print(f"✅ Val:   {len(eval_dataset):,} sequence")

In [ ]:
#@title 🏗️ BÖLÜM 6: Model Oluşturma (Sıfırdan)
print("\n" + "=" * 60)
print("🏗️ BÖLÜM 6: Model Oluşturma (Sıfırdan)")
print("=" * 60)

from transformers import LlamaForCausalLM, LlamaConfig

# H100 optimizasyonları
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# Flash Attention kontrolü
try:
    import flash_attn
    attn_impl = "flash_attention_2"
except ImportError:
    attn_impl = "sdpa"

config = LlamaConfig(
    vocab_size=VOCAB_SIZE,
    hidden_size=4096,
    intermediate_size=11008,
    num_hidden_layers=32,
    num_attention_heads=32,
    num_key_value_heads=8,
    max_position_embeddings=SEQ_LENGTH,
    rope_theta=500000.0,
    hidden_act="silu",
    rms_norm_eps=1e-5,
    tie_word_embeddings=False,
    torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl,
)

model = LlamaForCausalLM(config)
model = model.to(dtype=torch.bfloat16)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model oluşturuldu: {total_params/1e6:.1f}M parametre")
print(f"  Hidden: {config.hidden_size}, Layers: {config.num_hidden_layers}")
print(f"  Heads: {config.num_attention_heads}, KV Heads: {config.num_key_value_heads}")
print(f"  Attention: {attn_impl}")

param_mem = total_params * 2 / 1e9  # BF16
optim_mem = total_params * 2 / 1e9  # 8-bit Adam
print(f"  VRAM tahmini: ~{param_mem + optim_mem:.1f} GB (model + optimizer)")

In [ ]:
#@title 🚀 BÖLÜM 7: EğİTİM (2 Saat H100)
print("\n" + "=" * 60)
print("🚀 BÖLÜM 7: EğİTİM (2 Saat H100)")
print("=" * 60)

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import time
import math

BATCH_SIZE = 4
GRAD_ACCUM = 4
effective_batch_tokens = BATCH_SIZE * GRAD_ACCUM * SEQ_LENGTH
MAX_STEPS = 400  # 300-400 iterasyon yeterli, fazlası overfit yapar

print(f"  Efektif batch: {effective_batch_tokens:,} token")
print(f"  Max steps: {MAX_STEPS:,}")
print(f"  Toplam token: ~{MAX_STEPS * effective_batch_tokens / 1e6:.0f}M")

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_steps=40,
    weight_decay=0.1,
    max_grad_norm=1.0,
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    logging_first_step=True,
    save_steps=100,
    save_total_limit=5,
    eval_strategy="steps",
    eval_steps=100,
    report_to="none",
    run_name="llm-6b-tr-en-2h",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    optim="adamw_bnb_8bit",
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    seed=42,
    include_tokens_per_second=True,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

start_time = time.time()
print(f"\n🚀 Eğitim başlıyor... (Hedef: {MAX_STEPS} step)")
print("   Loss düşüşünü takip edin!")
print("-" * 60)

trainer.train()

elapsed = time.time() - start_time
print(f"\n✅ Eğitim tamamlandı! Süre: {elapsed/60:.1f} dakika ({elapsed/3600:.1f} saat)")

In [ ]:
#@title 💾 BÖLÜM 8: Model Kaydetme
print("\n" + "=" * 60)
print("💾 BÖLÜM 8: Model Kaydetme")
print("=" * 60)

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"✅ Model kaydedildi: {MODEL_DIR}")

import json

final_loss = "N/A"
for log_entry in reversed(trainer.state.log_history):
    if "loss" in log_entry:
        final_loss = log_entry["loss"]
        break

training_info = {
    "model_size": f"{total_params/1e6:.1f}M",
    "total_params": total_params,
    "training_steps": MAX_STEPS,
    "training_time_minutes": elapsed / 60,
    "total_tokens_processed": MAX_STEPS * effective_batch_tokens,
    "gpu": gpu_name,
    "final_loss": final_loss,
}
with open(os.path.join(MODEL_DIR, "training_info.json"), "w") as f:
    json.dump(training_info, f, indent=2)

print(f"✅ training_info.json kaydedildi")

In [ ]:
#@title 📝 BÖLÜM 9: Hızlı Değerlendirme
print("\n" + "=" * 60)
print("📝 BÖLÜM 9: Hızlı Değerlendirme")
print("=" * 60)

model.eval()

test_prompts = [
    "Türkiye'nin başkenti",
    "Istanbul is a city that",
    "Python programming language is",
    "Yapay zeka teknolojisi",
    "The transformer architecture",
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n🔹 {prompt}")
    print(f"   → {response}")

In [ ]:
#@title 🚀 BÖLÜM 10: HuggingFace Hub'a Yükleme
print("\n" + "=" * 60)
print("🚀 BÖLÜM 10: HuggingFace Hub'a Yükleme")
print("=" * 60)

# NOT: Token/secret değerlerini notebook içine gömmeyin.
HF_REPO_ID = os.getenv("HF_REPO_ID", "cebrailbagatarhan/turkmodel-6b")

try:
    from huggingface_hub import HfApi, login

    HF_TOKEN = os.getenv("HF_TOKEN")
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN bulunamadı (env/Secrets).")

    print("📝 HuggingFace Hub'a giriş yapılıyor...")
    login(token=HF_TOKEN)

    api = HfApi()

    try:
        api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, repo_type="model")
        print(f"  ✅ HF repo hazır: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"  ⚠️ Repo oluşturulamadı: {e}")

    print("📤 Model yükleniyor (bu birkaç dakika sürebilir)...")
    api.upload_folder(
        folder_path=MODEL_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message=f"TR-EN LLM - {MAX_STEPS} steps - {elapsed/60:.0f}min on {gpu_name}",
    )
    print(f"  ✅ Model yüklendi: https://huggingface.co/{HF_REPO_ID}")
    print(f"  📥 İndirmek için: git clone https://huggingface.co/{HF_REPO_ID}")

except Exception as e:
    print(f"  ⚠️ HuggingFace Hub yükleme atlanıyor: {e}")
    print(f"  → Manuel yükleme: huggingface-cli upload {HF_REPO_ID} ./final_model")

In [ ]:
#@title 📦 BÖLÜM 11: GitHub'a Push
print("\n" + "=" * 60)
print("📦 BÖLÜM 11: GitHub'a Push")
print("=" * 60)

# NOT: GitHub token'ı kod içine gömmeyin. Ortam değişkeni / Colab Secrets kullanın.
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    print("⚠️  GITHUB_TOKEN bulunamadı (env/Secrets). GitHub push atlanıyor.")
else:
    GITHUB_REPO = f"https://{GITHUB_TOKEN}@github.com/cebrailbagatarhan/turkmodel.git"

    try:
        subprocess.run(["git", "config", "--global", "user.email", "cebrailbagatarhan@users.noreply.github.com"], check=False)
        subprocess.run(["git", "config", "--global", "user.name", "cebrailbagatarhan"], check=False)

        REPO_DIR = "/content/turkmodel_repo"
        if os.path.exists(REPO_DIR):
            import shutil
            shutil.rmtree(REPO_DIR)

        subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)

        import shutil
        results_dir = os.path.join(REPO_DIR, "results")
        os.makedirs(results_dir, exist_ok=True)

        info_src = os.path.join(MODEL_DIR, "training_info.json")
        if os.path.exists(info_src):
            shutil.copy2(info_src, os.path.join(results_dir, "training_info.json"))

        eval_src = os.path.join(MODEL_DIR, "turkish_eval_results.json")
        if os.path.exists(eval_src):
            shutil.copy2(eval_src, os.path.join(results_dir, "turkish_eval_results.json"))

        readme_path = os.path.join(REPO_DIR, "llm-8b-tr-en", "TRAINING_RESULTS.md")
        os.makedirs(os.path.dirname(readme_path), exist_ok=True)
        with open(readme_path, "w", encoding="utf-8") as f:
            content = f"# Eğitim Sonuçları\n\n"
            content += "## Model Bilgileri\n"
            content += f"- **Boyut:** {total_params/1e6:.1f}M parametre\n"
            content += "- **Mimari:** Llama (Dense Transformer)\n"
            content += f"- **Eğitim:** {MAX_STEPS} step, {elapsed/60:.1f} dakika\n"
            content += f"- **GPU:** {gpu_name}\n"
            content += f"- **Final Loss:** {final_loss}\n\n"
            content += "## Modeli İndir\n"
            content += "```bash\n"
            content += "pip install huggingface_hub\n"
            content += f"huggingface-cli download {HF_REPO_ID} --local-dir ./model\n"
            content += "```\n"
            f.write(content)

        subprocess.run(["git", "-C", REPO_DIR, "add", "-A"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "commit", "-m",
                         f"Egitim sonuclari - {total_params/1e6:.0f}M model - {MAX_STEPS} steps"],
                        check=True)
        subprocess.run(["git", "-C", REPO_DIR, "push"], check=True)

        print("  ✅ GitHub'a push edildi!")
        print("  📄 Sonuçlar: results/training_info.json")

    except Exception as e:
        print(f"  ⚠️ GitHub push atlanıyor: {e}")
        print(f"  → Model Google Drive'da mevcut: {MODEL_DIR}")

In [ ]:
#@title 🎉 TAMAMLANDI!
print("\n" + "=" * 60)
print("🎉 TAMAMLANDI!")
print("=" * 60)
print(f"""\n📊 Özet:
  Model:     {total_params/1e6:.1f}M parametre (Llama mimarisi)
  Eğitim:    {MAX_STEPS} step, {elapsed/60:.1f} dakika
  GPU:       {gpu_name}
  Kaydedildi: {MODEL_DIR}

📥 Modeli İndirme Yolları:
  1. HuggingFace: https://huggingface.co/{HF_REPO_ID}
  2. Google Drive: {MODEL_DIR}

⏭️ Sonraki Adımlar:
  1. SFT (Supervised Fine-Tuning) yapabilirsiniz
  2. Daha fazla veri ile eğitime devam edebilirsiniz
  3. Daha büyük model (1B, 8B) için aynı pipeline'ı kullanın
""")